In [ ]:
import osmnx as ox
import networkx as nx
import numpy as np
from flask import Flask, request, jsonify
from flask_cors import CORS
import random
import math

app = Flask(__name__)
CORS(app)

# Configuração do grafo (baixado uma vez e cacheado)
# Escolha uma área razoável: exemplo, centro de São Paulo
# Para produção, você pode ajustar os limites
G = None
def obter_grafo():
    global G
    if G is None:
        # Baixa o grafo da região metropolitana de São Paulo (bounds)
        # Ajuste conforme necessário
        G = ox.graph_from_place('São Paulo, Brazil', network_type='drive', simplify=True)
    return G

def distancia_rodo(p1, p2, grafo):
    """Calcula a distância rodoviária em metros entre dois pontos (lat, lng)"""
    # Encontra os nós mais próximos
    orig_node = ox.distance.nearest_nodes(grafo, p1[1], p1[0])
    dest_node = ox.distance.nearest_nodes(grafo, p2[1], p2[0])
    # Rota mais curta
    try:
        route = nx.shortest_path(grafo, orig_node, dest_node, weight='length')
        # Soma dos comprimentos
        dist = sum(ox.utils_graph.get_route_edge_attributes(grafo, route, 'length'))
    except nx.NetworkXNoPath:
        # Fallback para distância euclidiana (caso não haja rota)
        dist = math.hypot(p1[0]-p2[0], p1[1]-p2[1]) * 111000  # aprox km
    return dist

def matriz_distancias(pontos, grafo):
    """Matriz de distâncias rodoviárias entre todos os pontos (incluindo depósito)"""
    n = len(pontos)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = distancia_rodo(pontos[i], pontos[j], grafo)
            D[i,j] = d
            D[j,i] = d
    return D

# ============================================
# ALGORITMO DE OTIMIZAÇÃO (Simulated Annealing)
# ============================================
def tsp_sa(matriz_dist, temp_inicial=1000, temp_final=0.1, alpha=0.99, max_iter=5000):
    n = matriz_dist.shape[0]
    # Solução inicial: ordem dos pontos (0 é depósito, depois clientes)
    sol = list(range(n))
    random.shuffle(sol[1:])  # randomiza clientes, mantém depósito em 0
    custo = lambda rota: sum(matriz_dist[rota[i], rota[(i+1)%n]] for i in range(n))
    
    melhor_sol = sol.copy()
    melhor_custo = custo(sol)
    
    T = temp_inicial
    while T > temp_final:
        for _ in range(max_iter):
            # Gerar vizinho: trocar dois clientes
            novo = sol.copy()
            i, j = random.sample(range(1, n), 2)
            novo[i], novo[j] = novo[j], novo[i]
            novo_custo = custo(novo)
            delta = novo_custo - custo(sol)
            if delta < 0 or random.random() < math.exp(-delta/T):
                sol = novo
                if novo_custo < melhor_custo:
                    melhor_sol = novo.copy()
                    melhor_custo = novo_custo
        T *= alpha
    return melhor_sol, melhor_custo

# ============================================
# BENCHMARK: Vizinho Mais Próximo (Greedy)
# ============================================
def tsp_vizinho_mais_proximo(matriz_dist):
    n = matriz_dist.shape[0]
    visitados = [False]*n
    rota = [0]  # começa no depósito
    visitados[0] = True
    atual = 0
    for _ in range(n-1):
        proximo = None
        melhor_dist = float('inf')
        for i in range(n):
            if not visitados[i] and matriz_dist[atual][i] < melhor_dist:
                melhor_dist = matriz_dist[atual][i]
                proximo = i
        if proximo is None:
            break
        rota.append(proximo)
        visitados[proximo] = True
        atual = proximo
    # Calcular custo total
    custo = sum(matriz_dist[rota[i]][rota[(i+1)%n]] for i in range(n))
    return rota, custo

# ============================================
# ROTA PARA COORDENADAS (para exibir no mapa)
# ============================================
def rota_para_coordenadas(rota_indices, pontos):
    """Retorna lista de [lat, lng] na ordem da rota"""
    return [pontos[i] for i in rota_indices]

# ============================================
# ENDPOINT
# ============================================
@app.route('/otimizar', methods=['POST'])
def otimizar():
    data = request.get_json()
    deposito = data['deposito']  # [lat, lng]
    clientes = data['clientes']  # lista de [lat, lng]
    
    # Monta lista de pontos: depósito primeiro
    pontos = [deposito] + clientes
    grafo = obter_grafo()
    
    # Matriz de distâncias reais
    D = matriz_distancias(pontos, grafo)
    
    # Otimização SA
    rota_sa, custo_sa = tsp_sa(D)
    coords_sa = rota_para_coordenadas(rota_sa, pontos)
    # Rota benchmark (vizinho mais próximo)
    rota_bench, custo_bench = tsp_vizinho_mais_proximo(D)
    coords_bench = rota_para_coordenadas(rota_bench, pontos)
    
    # Para o traçado das rotas, precisamos das coordenadas dos trechos (ruas)
    # Para simplificar, vamos retornar apenas as coordenadas dos pontos de parada
    # O frontend desenhará linhas retas entre eles, mas se quisermos seguir as ruas,
    # precisaríamos extrair a rota completa do grafo – isso é mais complexo.
    # Vamos retornar as coordenadas dos pontos de parada, o que já dá uma boa visualização.
    
    return jsonify({
        'rota_sa': coords_sa,
        'custo_sa': float(custo_sa),
        'rota_benchmark': coords_bench,
        'custo_benchmark': float(custo_bench),
        'diferenca_percentual': ((custo_bench - custo_sa) / custo_bench) * 100
    })

if __name__ == '__main__':
    app.run(debug=True, port=5000)